# Benchmarking Computacional – Grupo 04
## Fundamentos de Arquitectura de Computadoras, Sistemas Operativos y Redes

**Dataset utilizado:** OWID Energy Data – Our World in Data  
**Fuente:** https://github.com/owid/energy-data  
**Descripción:** Consumo energético histórico global (23.377 registros, 130 columnas, ~8.8 MB)  
**Librerías comparadas:** `pandas`, `polars`, `numba`

---
### Operaciones evaluadas
1. **Lectura** del CSV desde disco
2. **Filtrado** por año ≥ 2000
3. **Agregación** por país: promedio de generación eléctrica, emisiones GEI e intensidad de carbono
4. **Ordenamiento** y selección del top 20
5. **Cálculo numérico intensivo** (Numba): estimación de huella de carbono usando PUE y factor de emisión

In [ ]:
# Instalación de dependencias (ejecutar solo si es necesario)
# !pip install pandas polars numba numpy psutil memory-profiler matplotlib

In [ ]:
import pandas as pd
import polars as pl
import numpy as np
import numba as nb
import time
import tracemalloc
import psutil
import os
import json
import matplotlib.pyplot as plt
import urllib.request

print(f'pandas  {pd.__version__}')
print(f'polars  {pl.__version__}')
print(f'numba   {nb.__version__}')

In [ ]:
# Descarga del dataset público
CSV_PATH = 'owid-energy-data.csv'
if not os.path.exists(CSV_PATH):
    url = 'https://raw.githubusercontent.com/owid/energy-data/master/owid-energy-data.csv'
    print('Descargando dataset...')
    urllib.request.urlretrieve(url, CSV_PATH)
    print(f'Descargado: {os.path.getsize(CSV_PATH)/1024/1024:.2f} MB')
else:
    print(f'Dataset ya presente: {os.path.getsize(CSV_PATH)/1024/1024:.2f} MB')

# Vista previa
pd.read_csv(CSV_PATH, nrows=3)[['country','year','electricity_generation','greenhouse_gas_emissions','carbon_intensity_elec']]

In [ ]:
# ─── Función auxiliar de medición ───────────────────────────────────────────
PROCESS = psutil.Process(os.getpid())
resultados = {}

def medir(label, fn):
    """Mide tiempo de ejecución y pico de memoria."""
    tracemalloc.start()
    t0 = time.perf_counter()
    salida = fn()
    elapsed = time.perf_counter() - t0
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    resultados[label] = {
        'tiempo_s': round(elapsed, 4),
        'memoria_MB': round(peak / 1024**2, 2)
    }
    print(f'[{label:8s}]  tiempo = {elapsed:.4f} s  |  mem pico = {peak/1024**2:.2f} MB')
    return salida

## 1. Benchmark con Pandas

In [ ]:
def pandas_pipeline():
    """
    Pipeline pandas:
      1. Lectura completa del CSV
      2. Filtrado por año >= 2000
      3. GroupBy + agg (mean de 3 columnas)
      4. Sort + head(20)
    """
    df = pd.read_csv(CSV_PATH)
    df = df[df['year'] >= 2000]
    agg = df.groupby('country').agg(
        elec_promedio      = ('electricity_generation',  'mean'),
        ghg_promedio       = ('greenhouse_gas_emissions','mean'),
        intensidad_carbono = ('carbon_intensity_elec',   'mean'),
        n_registros        = ('year', 'count')
    ).reset_index()
    return agg.sort_values('elec_promedio', ascending=False).head(20)

resultado_pandas = medir('pandas', pandas_pipeline)
resultado_pandas.head(5)

## 2. Benchmark con Polars (Lazy Evaluation)

In [ ]:
def polars_pipeline():
    """
    Pipeline Polars con evaluación LAZY:
      - scan_csv() no lee nada hasta .collect()
      - El optimizador de consultas reordena y fusiona operaciones
      - Ejecución paralela multi-core (Rust interno)
    """
    df = pl.scan_csv(CSV_PATH)          # <-- LAZY: solo registra la intención
    return (
        df.filter(pl.col('year') >= 2000)
          .group_by('country')
          .agg([
              pl.col('electricity_generation').mean().alias('elec_promedio'),
              pl.col('greenhouse_gas_emissions').mean().alias('ghg_promedio'),
              pl.col('carbon_intensity_elec').mean().alias('intensidad_carbono'),
              pl.col('year').count().alias('n_registros')
          ])
          .sort('elec_promedio', descending=True)
          .limit(20)
          .collect()                    # <-- AQUÍ ejecuta todo el plan optimizado
    )

resultado_polars = medir('polars', polars_pipeline)
resultado_polars.head(5)

## 3. Benchmark con Numba (JIT + Paralelismo)

In [ ]:
@nb.njit(parallel=True)
def calcular_huella_carbono(consumo_arr, pue_arr, factor_co2):
    """
    Cálculo vectorizado de huella de carbono para N registros.
    
    Fórmula:
      kWh_total = consumo_TWh * pue * 8760 h/año
      kg_CO2    = kWh_total * factor_co2 (kg CO2 / kWh)
    
    nb.prange → paraleliza el loop en múltiples cores (evita el GIL)
    @njit     → compila a código máquina nativo (JIT) en la primera llamada
    """
    n = len(consumo_arr)
    resultado = np.zeros(n)
    for i in nb.prange(n):   # loop paralelo
        kwh = consumo_arr[i] * pue_arr[i] * 8760.0
        resultado[i] = kwh * factor_co2
    return resultado

# Warm-up: primera llamada compila la función (no se mide)
_w = calcular_huella_carbono(
    np.array([1.0, 2.0]), np.array([1.2, 1.5]), 0.233
)
print('JIT compilado (warm-up completado)')

def numba_pipeline():
    """
    1. Lectura con pandas (carga de datos)
    2. Extracción de arrays numpy
    3. Cálculo intensivo paralelo con Numba
    """
    df = pd.read_csv(CSV_PATH)
    df = df[df['year'] >= 2000].dropna(subset=['electricity_generation'])
    consumo = df['electricity_generation'].values.astype(np.float64)
    # Simular variación de PUE entre 1.1 (eficiente) y 1.8 (ineficiente)
    pue = np.random.uniform(1.1, 1.8, len(consumo))
    factor_co2 = 0.233  # kg CO₂/kWh – Red eléctrica global promedio (IEA, 2023)
    return calcular_huella_carbono(consumo, pue, factor_co2)

resultado_numba = medir('numba', numba_pipeline)
print(f'\nRegistros procesados: {len(resultado_numba):,}')
print(f'Huella promedio estimada: {resultado_numba.mean():.1f} kg CO₂ / TWh')

## 4. Resultados comparativos y gráficos

In [ ]:
# ── Tabla resumen ──────────────────────────────────────────────────────────
df_res = pd.DataFrame(resultados).T.reset_index()
df_res.columns = ['Librería', 'Tiempo (s)', 'Mem pico (MB)']
print(df_res.to_string(index=False))

# ── Gráfico comparativo ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
libs = list(resultados.keys())
tiempos = [resultados[l]['tiempo_s']    for l in libs]
memorias = [resultados[l]['memoria_MB'] for l in libs]
colores = ['#4C72B0', '#55A868', '#C44E52']

# Tiempo
bars0 = axes[0].bar(libs, tiempos, color=colores, edgecolor='white', linewidth=0.8)
axes[0].set_title('Tiempo de ejecución (s)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Segundos')
for bar, val in zip(bars0, tiempos):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{val:.4f}s', ha='center', va='bottom', fontsize=10)

# Memoria
bars1 = axes[1].bar(libs, memorias, color=colores, edgecolor='white', linewidth=0.8)
axes[1].set_title('Pico de memoria (MB)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('MB')
for bar, val in zip(bars1, memorias):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{val:.2f} MB', ha='center', va='bottom', fontsize=10)

plt.suptitle('Benchmarking: pandas vs polars vs numba\nDataset: OWID Energy Data (23.377 registros)',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('benchmark_resultados.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico guardado como benchmark_resultados.png')

## 5. Cálculo de huella de carbono – 1 TB en la nube durante 1 año

In [ ]:
# ── Parámetros ──────────────────────────────────────────────────────────────
consumo_hdd_W_por_TB = 6.0      # W por TB: HDD empresarial (Seagate Exos 7E10)
consumo_ssd_W_por_TB = 1.2      # W por TB: SSD NVMe empresarial (estimado)
horas_anio = 8760               # h/año
pue_global = 1.58               # PUE promedio global histórico (Uptime Institute, 2023)
pue_google = 1.10               # PUE Google 2023 (Google Sustainability Report)
pue_propuesta = 1.15            # PUE estimado para el DC propuesto (Patagonia)

# Factor de emisión: kg CO₂ por kWh
factor_arg = 0.340              # Argentina (CAMMESA, 2023)
factor_global = 0.233           # Promedio global (IEA, 2023)
factor_renovable = 0.030        # Energía eólica (estimación ciclo de vida)

print('='*60)
print('CÁLCULO DE HUELLA DE CARBONO – 1 TB en la nube / año')
print('='*60)

for nombre, consumo_W in [('HDD empresarial', consumo_hdd_W_por_TB), 
                           ('SSD NVMe',       consumo_ssd_W_por_TB)]:
    for pue_nombre, pue in [('PUE global (1.58)', pue_global),
                             ('PUE Google (1.10)', pue_google),
                             ('PUE propuesto Patagonia (1.15)', pue_propuesta)]:
        kwh = (consumo_W * horas_anio * pue) / 1000
        for factor_nombre, factor in [('Red argentina', factor_arg),
                                       ('Red global',   factor_global),
                                       ('Eólica',       factor_renovable)]:
            kg_co2 = kwh * factor
            if nombre == 'HDD empresarial' and 'google' in pue_nombre.lower() and 'global' in factor_nombre.lower():
                print(f'\n★ Referencia principal ({nombre}, {pue_nombre}, {factor_nombre})')
                print(f'  Consumo: {consumo_W} W × {horas_anio} h × PUE {pue} = {kwh:.2f} kWh/año')
                print(f'  Emisión: {kwh:.2f} kWh × {factor} kg CO₂/kWh = {kg_co2:.2f} kg CO₂/año')

# Tabla resumen limpia
print('\n' + '-'*60)
print(f'{"Escenario":<35} {"kWh/año":>10} {"kg CO₂/año":>12}')
print('-'*60)
escenarios = [
    ('HDD + PUE 1.58 + red argentina',  consumo_hdd_W_por_TB, pue_global,    factor_arg),
    ('HDD + PUE 1.10 + red global',     consumo_hdd_W_por_TB, pue_google,    factor_global),
    ('HDD + PUE 1.15 + eólica (prop.)', consumo_hdd_W_por_TB, pue_propuesta, factor_renovable),
    ('SSD + PUE 1.10 + red global',     consumo_ssd_W_por_TB, pue_google,    factor_global),
    ('SSD + PUE 1.15 + eólica (prop.)', consumo_ssd_W_por_TB, pue_propuesta, factor_renovable),
]
for nombre, w, pue, factor in escenarios:
    kwh = (w * horas_anio * pue) / 1000
    kg  = kwh * factor
    print(f'{nombre:<35} {kwh:>10.2f} {kg:>12.3f}')
print('-'*60)

## 6. Análisis y conclusiones del benchmarking

### Resultados observados

| Librería | Tiempo (s) | Mem pico (MB) | Observación |
|----------|-----------|---------------|-------------|
| **pandas** | ~0.33 | ~31 | Evaluación eager, carga todo en RAM |
| **polars** | ~0.04 | ~0.02 | Lazy + paralelo en Rust → **8.7× más rápido** |
| **numba**  | ~0.58 | ~37 | Incluye lectura pandas + cálculo JIT paralelo |

### Justificación teórica

**Polars y evaluación lazy:** Polars usa un *query optimizer* que analiza el grafo de operaciones antes de ejecutar cualquiera. Al usar `scan_csv()` en lugar de `read_csv()`, el motor puede pushdown predicados (el filtro `year >= 2000` se aplica *durante* la lectura, no después), fusionar pasos y asignar columnas a múltiples cores de forma nativa. Esto conecta con el concepto de **localidad espacial**: Polars organiza los datos en formato columnar, lo que permite que las líneas de caché del procesador sean aprovechadas con mayor eficiencia al procesar columnas contiguas en memoria.

**Numba y compilación JIT:** La función `calcular_huella_carbono` usa el decorador `@nb.njit(parallel=True)`. En la primera llamada (warm-up), Numba compila la función a código máquina nativo, saltando el *Global Interpreter Lock* (GIL) de Python. El uso de `nb.prange` divide el loop entre todos los núcleos disponibles, logrando **paralelismo multi-core** verdadero. El overhead medido corresponde principalmente a la lectura del CSV con pandas; el cálculo numérico puro es extremadamente rápido.

**Pandas y evaluación eager:** Pandas carga el CSV completo en memoria (~31 MB de pico), aplica los filtros sobre los datos ya cargados y ejecuta el GroupBy de forma secuencial. Es la solución más sencilla pero menos eficiente para datasets grandes.